# Depth + Segmentation → Colored 3D Point Cloud

Take one driving image, run a monocular depth model and a semantic segmentation
model on it, then lift everything into a 3D point cloud colored by class.


In [ ]:
!pip install -q transformers torch pillow opencv-python plotly matplotlib

In [ ]:
import numpy as np
import cv2
import torch
from PIL import Image
from transformers import pipeline
import matplotlib.pyplot as plt
import plotly.graph_objects as go

IMAGE_PATH = "data/left/000009.png"
DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Small-hf"
SEG_MODEL = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
INPUT_SIZE = (640, 192)
FX = 500
FY = 500
DEVICE = 0 if torch.cuda.is_available() else -1

In [ ]:
image = Image.open(IMAGE_PATH).convert("RGB")
plt.figure(figsize=(12, 4))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
depth_pipe = pipeline("depth-estimation", model=DEPTH_MODEL, device=DEVICE)
depth_out = depth_pipe(image)
depth = np.array(depth_out["depth"]).astype("float32")

plt.figure(figsize=(12, 4))
plt.imshow(depth, cmap="magma")
plt.axis("off")
plt.show()
cv2.imwrite("depth.png", depth.astype("uint8"))

In [ ]:
CITYSCAPES_COLORS = {
    "road": (128, 64, 128), "sidewalk": (244, 35, 232), "building": (70, 70, 70),
    "wall": (102, 102, 156), "fence": (190, 153, 153), "pole": (153, 153, 153),
    "traffic light": (250, 170, 30), "traffic sign": (220, 220, 0),
    "vegetation": (107, 142, 35), "terrain": (152, 251, 152), "sky": (70, 130, 180),
    "person": (220, 20, 60), "rider": (255, 0, 0), "car": (0, 0, 142),
    "truck": (0, 0, 70), "bus": (0, 60, 100), "train": (0, 80, 100),
    "motorcycle": (0, 0, 230), "bicycle": (119, 11, 32),
}

seg_pipe = pipeline("image-segmentation", model=SEG_MODEL, device=DEVICE)
seg_out = seg_pipe(image)

seg_color = np.zeros((image.size[1], image.size[0], 3), dtype="uint8")
for seg in seg_out:
    mask = np.array(seg["mask"]) > 0
    seg_color[mask] = CITYSCAPES_COLORS.get(seg["label"], (0, 0, 0))

plt.figure(figsize=(12, 4))
plt.imshow(seg_color)
plt.axis("off")
plt.show()

In [ ]:
depth_r = cv2.resize(depth, INPUT_SIZE)
seg_r = cv2.resize(seg_color, INPUT_SIZE, interpolation=cv2.INTER_NEAREST)

H, W = depth_r.shape
cx, cy = W / 2, H / 2
u, v = np.meshgrid(np.arange(W), np.arange(H))

z = depth_r
x = (u - cx) * z / FX
y = (v - cy) * z / FY

points = np.stack([x, y, z], axis=-1).reshape(-1, 3)
colors = seg_r.reshape(-1, 3)
np.save("pointcloud.npy", np.concatenate([points, colors], axis=1))
print(points.shape)

In [ ]:
idx = np.random.choice(len(points), 50000, replace=False)
p = points[idx]
c = colors[idx]
rgb = ["rgb(%d,%d,%d)" % (r, g, b) for r, g, b in c]

fig = go.Figure(go.Scatter3d(
    x=p[:, 0], y=-p[:, 1], z=p[:, 2],
    mode="markers",
    marker=dict(size=1.4, color=rgb, opacity=1.0),
))
fig.update_layout(
    template="plotly_dark",
    width=1000, height=650,
    margin=dict(l=0, r=0, t=0, b=0),
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        camera=dict(eye=dict(x=0.0, y=-0.35, z=-1.4), up=dict(x=0, y=1, z=0)),
    ),
)
fig.show()